# 03 — Discounting & Pricing

**Core question:** does discounting buy retention, or just give up margin?

**The confound that makes this hard:** discounts may be offered *to* users who are already
likely to churn (retention offers), which would reverse the apparent sign of a naive
correlation. If KKBox routes discounts toward at-risk users, a naive "discount depth vs.
renewal rate" chart can show deeper discounts associated with *lower* renewal — not because
discounting doesn't work, but because the discounted population was riskier to begin with. This
notebook checks for that confound explicitly rather than reporting the naive number as a finding.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src import pricing, plotting

plotting.set_style()
periods = pd.read_parquet("../data/interim/periods.parquet")

## Realized discount depth
`discount_depth = (plan_list_price - actual_amount_paid) / plan_list_price`, computed in `src/cohorts.py`. Rows with `plan_list_price == 0` are excluded — a discount off a $0 list price is undefined, not zero.

In [ ]:
priced = periods[periods["plan_list_price"] > 0]
print(priced["discount_depth"].describe().round(3))
print(f"\n{(priced['discount_depth'] <= 0.001).mean():.1%} of priced transactions were at full list price.")

## Naive correlation (descriptive only — do not read this as an effect)

In [ ]:
naive_summary, naive_test = pricing.renewal_rate_by_discount_bucket(priced)
print(naive_summary)
print(f"\nchi2 = {naive_test['chi2']:.1f}, dof = {naive_test['dof']}, p = {naive_test['p_value']:.4g}")

fig, ax = plotting.plot_ordinal_bars(
    naive_summary["discount_bucket"].astype(str).tolist(),
    naive_summary["renewal_rate"].tolist(),
    "Renewal rate by discount depth (naive, unadjusted)",
    "Renewal rate",
    save_path="../figures/03_naive_discount.png",
)

**This chart is descriptive, not causal — read the next two sections before drawing a
conclusion from it.** If deeper discounts show *lower* renewal, the obvious headline
("discounting doesn't work, cut it") could be backwards: KKBox may be routing discounts to
users it already expects to churn, in which case the discounted group was riskier to begin
with and the naive comparison understates whatever positive effect the discount actually has.

## Confound check 1: is the discount targeted at at-risk users?

If discount depth is significantly higher immediately after a voluntary cancellation on the
user's prior transaction, that's direct evidence discounts are being used as retention offers
— which would bias the naive comparison above.

In [ ]:
targeting_summary, targeting_test = pricing.discount_depth_by_prior_cancel(priced)
print(targeting_summary)
print(f"\nMann-Whitney U = {targeting_test['u_statistic']:.0f}, p = {targeting_test['p_value']:.4g}")

If `p_value` is small and the `prior_voluntary_cancel` group's median discount is meaningfully higher, the targeting confound is real and the naive chart above should not be read at face value. If the two groups look similar, that particular targeting mechanism isn't driving the naive result — though other, unobserved targeting (e.g. a support agent's manual judgment call, which isn't in this data) still can't be ruled out.

## Confound check 2: within-plan comparison
Holding `payment_plan_days` and `plan_list_price` fixed removes confounding from users self-selecting into different plan types (e.g. annual-plan buyers are already a different population than monthly).

In [ ]:
stratified = pricing.renewal_rate_by_discount_bucket_stratified(priced)
top_strata = (
    stratified.groupby(["payment_plan_days", "plan_list_price"])["n_periods"].sum()
    .sort_values(ascending=False).head(3).index
)
for plan_days, list_price in top_strata:
    sub = stratified[(stratified["payment_plan_days"] == plan_days) & (stratified["plan_list_price"] == list_price)]
    print(f"\nplan_list_price={list_price}, payment_plan_days={plan_days} (n={sub['n_periods'].sum():,})")
    print(sub[["discount_bucket", "n_periods", "renewal_rate"]].to_string(index=False))

## Adjusted view: logistic regression
Controls for plan length, list price, auto-renew status, and the prior-cancel proxy at once. Still not causal (see `src/pricing.py` docstring) — discount depth is not randomly assigned — but a better-adjusted estimate than the naive chart above.

In [ ]:
model = pricing.fit_retention_logit(priced)
print(model.summary())

In [ ]:
coef = model.params["discount_depth"]
p_val = model.pvalues["discount_depth"]
odds_ratio = np.exp(coef)
print(f"discount_depth coefficient: {coef:.3f} (p={p_val:.4g}), odds ratio {odds_ratio:.3f}")
print()
print("Interpretation: holding plan length, list price, auto-renew, and the prior-cancel proxy")
print("constant, this is the association between discount depth and renewal odds. Report the")
print("sign and the p-value together with the caveat that unobserved targeting (manual retention")
print("offers not captured by prior_is_cancel) can still bias this estimate.")

## Section summary

- State plainly whether the naive and adjusted results agree in sign after running this
  notebook.
- If they diverge, that divergence **is** the finding: it means the targeting confound is
  material, and "targeted discounts to at-risk users appear to retain them" is a very different
  recommendation from "discounting broadly retains users" — don't collapse the two.
- If discounting shows no measurable retention effect once confounds are controlled for, report
  that. A clean null result here is a better portfolio piece than a manufactured one.